# 공공기관식 문화시설 접근성 지표 재구현
- 목적: 정부·공공기관에서 사용하는 최근접 시설 접근성 지표를 문화누리카드 분석 데이터에 적용
- 계산권역: 서울 + 서울 경계 외부 25km 경쟁권 도로망·가맹점
- 결과대상: 서울 100m 격자, 서울 행정동, 서울 시군구
- 산출지표: 중분류별 최근접 문화시설 도로거리, 10km 서비스권역 내 문화누리대상자 비율


## 지표 설계
- 정부식 접근성 지표는 격자에서 가장 가까운 문화시설까지의 도로망 최단거리로 계산한다.
- 공연·문화시설은 지역거점시설 기준을 적용하여 차량 20분·10km 서비스권역으로 해석한다.
- 원 지표의 인구비율은 전체 거주인구 기준이나, 본 프로젝트에서는 정책 대상자인 문화누리대상자 추정인구를 기준으로 재산정한다.

$$
A_{i,k}=\min_{j \in C_k}(d_{i,n_i}+d_{n_i,n_j}+d_{n_j,j})
$$

$$
R_{r,k}=\frac{\sum_{i \in r}P_i \cdot \mathbf{1}(A_{i,k}\leq 10{,}000)}{\sum_{i \in r}P_i}\times100
$$


In [ ]:

import warnings
warnings.filterwarnings("ignore")

import heapq
from pathlib import Path

import geopandas as gpd
import networkx as nx
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


## 경로 설정
- 최신 경쟁권 분석 자료를 기준으로 입력 경로를 설정한다.
- 산출물은 access 폴더 내부 OUTPUT 하위에 별도 폴더로 저장한다.


In [ ]:

BASE_PATH = Path.cwd().resolve()

if BASE_PATH.name == "access":
    PROJECT_PATH = BASE_PATH.parent.parent
elif BASE_PATH.name == "notebooks":
    PROJECT_PATH = BASE_PATH.parent
elif BASE_PATH.name == "oracle_mnc_project":
    PROJECT_PATH = BASE_PATH
else:
    PROJECT_PATH = next(
        (path for path in [BASE_PATH, *BASE_PATH.parents] if (path / "notebooks").exists()),
        BASE_PATH,
    )

DATA_INPUT_PATH = PROJECT_PATH / "analysis_table" / "data" / "input"
DATA_OUTPUT_PATH = PROJECT_PATH / "analysis_table" / "data" / "output"
NETWORK_COMP_PATH = DATA_OUTPUT_PATH / "network_competition_25km"
KTDB_PATH = DATA_INPUT_PATH / "network" / "ktdb_transport_network"

GRID_PATH = NETWORK_COMP_PATH / "경쟁권25km_분석격자.parquet"
STORE_PATH = NETWORK_COMP_PATH / "경쟁권25km_분석가맹점.parquet"
ANALYSIS_AREA_PATH = NETWORK_COMP_PATH / "경쟁권25km_분석권역.gpkg"
SEOUL_MNC_PATH = DATA_OUTPUT_PATH / "서울시_100m_문화누리추정인구수.gpkg"
EXT_MNC_PATH = DATA_OUTPUT_PATH / "인천경기_외부25km_100m_문화누리대상자_추정인구.gpkg"

NODE_PATH = next(KTDB_PATH.rglob("2025node.txt"))
LINK_PATH = next(KTDB_PATH.rglob("2025link.txt"))

ACCESS_PATH = PROJECT_PATH / "notebooks" / "access"
NOTEBOOK_OUTPUT_PATH = ACCESS_PATH / "OUTPUT" / "public_access_index_25km"
NOTEBOOK_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

SERVICE_LIMIT_M = 10_000
ROAD_BUFFER_M = 2_000
ROAD_LINK_TYPES = list(range(101, 109))

facility_mid_categories = [
    "도서", "문화체험", "음악", "영상", "체육시설",
    "체육용품", "미술", "공연", "스포츠관람", "관광지"
]

print("프로젝트 경로:", PROJECT_PATH)
print("저장 경로:", NOTEBOOK_OUTPUT_PATH)
print("격자 경로:", GRID_PATH)
print("가맹점 경로:", STORE_PATH)


## 데이터 불러오기
- 분석권역 격자와 가맹점은 서울 + 외부 25km 자료를 사용한다.
- 문화누리대상자 추정인구는 서울과 외부권역 산출물을 결합해 붙인다.
- 최종 결과는 서울 격자만 산출하되, 최근접 시설 탐색에는 외부권역 가맹점까지 포함한다.


In [ ]:

grid = gpd.read_parquet(GRID_PATH).to_crs("EPSG:5179")
store = gpd.read_parquet(STORE_PATH).to_crs("EPSG:5179")

seoul_mnc = gpd.read_file(SEOUL_MNC_PATH)[["GRID_CD", "문화누리대상자_추정_인구수"]]
ext_mnc = gpd.read_file(EXT_MNC_PATH)[["GRID_CD", "문화누리대상자_추정_인구수"]]

mnc_pop = pd.concat([seoul_mnc, ext_mnc], ignore_index=True)
mnc_pop["문화누리대상자_추정_인구수"] = pd.to_numeric(
    mnc_pop["문화누리대상자_추정_인구수"],
    errors="coerce"
).fillna(0).round().astype(int)

mnc_pop = (
    mnc_pop
    .groupby("GRID_CD", as_index=False)["문화누리대상자_추정_인구수"]
    .sum()
)

grid = grid.drop(columns="문화누리대상자_추정_인구수", errors="ignore")
grid = grid.merge(mnc_pop, on="GRID_CD", how="left")
grid["문화누리대상자_추정_인구수"] = grid["문화누리대상자_추정_인구수"].fillna(0).astype(int)
grid["추정_인구수"] = pd.to_numeric(grid["추정_인구수"], errors="coerce").fillna(0).round().astype(int)

store_facility = store[store["중분류"].isin(facility_mid_categories)].copy()
store_facility = store_facility.dropna(subset=["geometry", "중분류"]).copy()

grid_seoul = grid[grid["서울여부"] == True].copy()

print("전체 분석권역 격자 구조:", grid.shape)
print("서울 결과대상 격자 구조:", grid_seoul.shape)
print("전체 분석권역 가맹점 구조:", store_facility.shape)
print("격자 GRID_CD 중복:", grid["GRID_CD"].duplicated().sum())
print("문화누리대상자 결측:", grid["문화누리대상자_추정_인구수"].isna().sum())
print("서울 문화누리대상자 추정인구 합:", grid_seoul["문화누리대상자_추정_인구수"].sum())
print("전체권역 문화누리대상자 추정인구 합:", grid["문화누리대상자_추정_인구수"].sum())

print("\n분석권역 시도별 격자 수")
print(grid["시도"].value_counts(dropna=False))
print("\n중분류별 가맹점 수")
print(store_facility["중분류"].value_counts().reindex(facility_mid_categories).fillna(0).astype(int))

display(grid_seoul.head())
display(store_facility.head())


## KTDB 도로망 불러오기
- KTDB node/link txt 첫 줄은 메타 정보이므로 제외한다.
- 일반 도로 링크 유형 101~108만 사용한다.
- 링크 길이는 km에서 m로 변환한다.


In [ ]:

node = pd.read_csv(
    NODE_PATH,
    sep=r"\s+",
    skiprows=1,
    header=None,
    names=["record_type", "node_id", "x", "y"],
    engine="python"
).drop(columns="record_type")

link = pd.read_csv(
    LINK_PATH,
    sep=r"\s+",
    skiprows=1,
    header=None,
    names=[
        "record_type", "from_node", "to_node", "length_km", "mode_code",
        "link_type", "lane", "capacity", "speed", "vdf", "cost"
    ],
    engine="python"
).drop(columns="record_type")

node["node_id"] = pd.to_numeric(node["node_id"], errors="coerce")
node["x"] = pd.to_numeric(node["x"], errors="coerce")
node["y"] = pd.to_numeric(node["y"], errors="coerce")
node = node.dropna(subset=["node_id", "x", "y"]).copy()
node["node_id"] = node["node_id"].astype("int64")

for col in ["from_node", "to_node", "length_km", "link_type", "lane", "capacity", "speed", "vdf", "cost"]:
    link[col] = pd.to_numeric(link[col], errors="coerce")

link = link.dropna(subset=["from_node", "to_node", "length_km", "link_type"]).copy()
link["from_node"] = link["from_node"].astype("int64")
link["to_node"] = link["to_node"].astype("int64")
link["link_type"] = link["link_type"].astype("int64")
link["length_m"] = link["length_km"] * 1000

print("KTDB node 구조:", node.shape)
print("KTDB link 구조:", link.shape)
print("node_id 중복:", node["node_id"].duplicated().sum())
print("from_node-to_node 중복:", link[["from_node", "to_node"]].duplicated().sum())
print("length_m 0 이하:", (link["length_m"] <= 0).sum())
print("\nlink_type 분포")
print(link["link_type"].value_counts().sort_index())


## 도로망 분석권역 필터링
- 서울 경계가 아니라 서울 + 외부 25km 분석권역을 기준으로 도로망을 자른다.
- 경계부 스냅 누락을 줄이기 위해 분석권역에 2km 여유 버퍼를 적용한다.


In [ ]:

ktdb_crs = (
    "+proj=tmerc +lat_0=38 +lon_0=128 +k=0.9999 "
    "+x_0=400000 +y_0=600000 +ellps=bessel "
    "+towgs84=-146.43,507.89,681.46 +units=m +no_defs"
)

node_gdf = gpd.GeoDataFrame(
    node.copy(),
    geometry=gpd.points_from_xy(node["x"], node["y"]),
    crs=ktdb_crs
).to_crs("EPSG:5179")

analysis_area = gpd.read_file(ANALYSIS_AREA_PATH).to_crs("EPSG:5179").dissolve()[["geometry"]].reset_index(drop=True)
analysis_area_buffer = analysis_area.copy()
analysis_area_buffer["geometry"] = analysis_area_buffer.geometry.buffer(ROAD_BUFFER_M)

node_in_buffer = gpd.sjoin(
    node_gdf,
    analysis_area_buffer[["geometry"]],
    how="inner",
    predicate="within"
).drop(columns="index_right", errors="ignore")

link_road = link[
    (link["length_m"] > 0) &
    (link["link_type"].isin(ROAD_LINK_TYPES))
].copy()

buffer_node_ids = set(node_in_buffer["node_id"])
link_road_area = link_road[
    link_road["from_node"].isin(buffer_node_ids) |
    link_road["to_node"].isin(buffer_node_ids)
].copy()

road_node_ids = set(link_road_area["from_node"]) | set(link_road_area["to_node"])
node_road_area = node_gdf[node_gdf["node_id"].isin(road_node_ids)].copy()

print("분석권역+2km 버퍼 내부 node 수:", f"{len(node_in_buffer):,}")
print("도로망 분석용 node 수:", f"{len(node_road_area):,}")
print("일반 도로 link_type 101~108 필터 후 link 수:", f"{len(link_road):,}")
print("분석권역 도로망 link 수:", f"{len(link_road_area):,}")
print("\n분석권역 도로망 link_type 분포")
print(link_road_area["link_type"].value_counts().sort_index())


## 도로망 그래프 구성
- 도로망은 무방향 그래프로 구성한다.
- 동일한 node 쌍이 여러 번 등장하면 가장 짧은 링크 길이를 대표 edge로 사용한다.
- 연결 성분은 도로망이 서로 끊어진 묶음이다.


In [ ]:

road_graph = nx.Graph()

for row in link_road_area.itertuples(index=False):
    from_node = int(row.from_node)
    to_node = int(row.to_node)
    length_m = float(row.length_m)
    current_edge = road_graph.get_edge_data(from_node, to_node)

    if current_edge is None or length_m < current_edge["length"]:
        road_graph.add_edge(from_node, to_node, length=length_m)

connected_components = sorted(nx.connected_components(road_graph), key=len, reverse=True)
node_to_component = {}
component_size = {}

for component_id, component_nodes in enumerate(connected_components):
    component_size[component_id] = len(component_nodes)
    for node_id in component_nodes:
        node_to_component[node_id] = component_id

print("도로망 그래프 node 수:", f"{road_graph.number_of_nodes():,}")
print("도로망 그래프 edge 수:", f"{road_graph.number_of_edges():,}")
print("그래프 연결 성분 수:", f"{len(connected_components):,}")
print("상위 연결 성분 크기:", [len(component) for component in connected_components[:10]])


## 격자·가맹점 도로망 노드 스냅
- 서울 격자 중심점과 전체권역 가맹점을 가장 가까운 도로망 node에 연결한다.
- 접근거리에는 격자-노드 스냅거리, 도로망 최단거리, 시설-노드 스냅거리를 모두 포함한다.


In [ ]:

node_snap_base = node_road_area[node_road_area["node_id"].isin(road_graph.nodes)].copy().reset_index(drop=True)
node_xy = np.column_stack([node_snap_base.geometry.x.to_numpy(), node_snap_base.geometry.y.to_numpy()])
node_tree = cKDTree(node_xy)
node_id_array = node_snap_base["node_id"].to_numpy()

grid_snap = grid_seoul.copy().to_crs("EPSG:5179")
grid_xy = np.column_stack([grid_snap["중심점_x"].to_numpy(), grid_snap["중심점_y"].to_numpy()])
grid_distance, grid_nearest_pos = node_tree.query(grid_xy, k=1)
grid_snap["도로망_노드ID"] = node_id_array[grid_nearest_pos]
grid_snap["격자_스냅거리_m"] = grid_distance

store_facility_snap = store_facility.copy().to_crs("EPSG:5179")
store_xy = np.column_stack([store_facility_snap.geometry.x.to_numpy(), store_facility_snap.geometry.y.to_numpy()])
store_distance, store_nearest_pos = node_tree.query(store_xy, k=1)
store_facility_snap["도로망_노드ID"] = node_id_array[store_nearest_pos]
store_facility_snap["시설_스냅거리_m"] = store_distance

grid_snap["도로망_연결성분ID"] = grid_snap["도로망_노드ID"].map(node_to_component)
grid_snap["도로망_연결성분_node수"] = grid_snap["도로망_연결성분ID"].map(component_size)
store_facility_snap["도로망_연결성분ID"] = store_facility_snap["도로망_노드ID"].map(node_to_component)

facility_component_count = (
    store_facility_snap
    .groupby(["중분류", "도로망_연결성분ID"], as_index=False)["도로망_노드ID"]
    .nunique()
    .rename(columns={"도로망_노드ID": "도로망_연결성분_시설노드수"})
)

print("서울 격자 도로망 node 결측:", grid_snap["도로망_노드ID"].isna().sum())
print("전체권역 문화시설 도로망 node 결측:", store_facility_snap["도로망_노드ID"].isna().sum())
print("\n격자 스냅거리 통계")
print(grid_snap["격자_스냅거리_m"].describe())
print("\n문화시설 스냅거리 통계")
print(store_facility_snap["시설_스냅거리_m"].describe())
print("격자 스냅거리 500m 초과:", (grid_snap["격자_스냅거리_m"] > 500).sum())
print("문화시설 스냅거리 300m 초과:", (store_facility_snap["시설_스냅거리_m"] > 300).sum())


## 중분류별 최근접 시설 도로거리 계산
- 각 중분류 시설 node를 출발점으로 하는 다중 출발점 Dijkstra를 수행한다.
- 최종 접근거리는 격자 스냅거리 + 도로망 비용 + 시설 스냅거리로 구성한다.


In [ ]:

def nearest_facility_dijkstra(graph, source_cost, cutoff):
    distance = {}
    source_node = {}
    heap = []

    for node_id, cost in source_cost.items():
        if pd.isna(cost):
            continue
        node_id = int(node_id)
        cost = float(cost)
        if cost > cutoff:
            continue
        if node_id not in distance or cost < distance[node_id]:
            distance[node_id] = cost
            source_node[node_id] = node_id
            heapq.heappush(heap, (cost, node_id, node_id))

    while heap:
        current_distance, current_node, origin_node = heapq.heappop(heap)
        if current_distance != distance.get(current_node):
            continue
        for next_node, edge_data in graph[current_node].items():
            next_distance = current_distance + float(edge_data["length"])
            if next_distance > cutoff:
                continue
            if next_node not in distance or next_distance < distance[next_node]:
                distance[next_node] = next_distance
                source_node[next_node] = origin_node
                heapq.heappush(heap, (next_distance, next_node, origin_node))

    return distance, source_node


grid_base_columns = [
    "GRID_CD", "시도", "서울여부", "행정동코드", "시군구", "행정동",
    "중심점_x", "중심점_y", "추정_인구수", "문화누리대상자_추정_인구수",
    "GRID_CD_500", "도로망_노드ID", "도로망_연결성분ID", "도로망_연결성분_node수",
    "격자_스냅거리_m", "geometry"
]

category_grid_tables = []
category_summary_rows = []

for category in facility_mid_categories:
    category_facility = store_facility_snap[store_facility_snap["중분류"] == category].copy()
    category_node_best = (
        category_facility
        .sort_values(["도로망_노드ID", "시설_스냅거리_m", "가맹점_ID"])
        .drop_duplicates("도로망_노드ID")
        .set_index("도로망_노드ID")
    )

    source_cost = category_node_best["시설_스냅거리_m"].to_dict()
    node_to_facility_cost, node_to_facility_node = nearest_facility_dijkstra(road_graph, source_cost, SERVICE_LIMIT_M)

    grid_category = grid_snap[grid_base_columns].copy()
    grid_category["중분류"] = category
    grid_category["노드_시설접근비용_m"] = grid_category["도로망_노드ID"].map(node_to_facility_cost)
    grid_category["최근접_시설노드ID"] = grid_category["도로망_노드ID"].map(node_to_facility_node)
    grid_category["최근접_가맹점_ID"] = grid_category["최근접_시설노드ID"].map(category_node_best["가맹점_ID"])
    grid_category["최근접_가맹점명"] = grid_category["최근접_시설노드ID"].map(category_node_best["가맹점명"])
    grid_category["최근접_가맹점_시도"] = grid_category["최근접_시설노드ID"].map(category_node_best["시도"])
    grid_category["최근접_가맹점_시군구"] = grid_category["최근접_시설노드ID"].map(category_node_best["시군구"])
    grid_category["시설_스냅거리_m"] = grid_category["최근접_시설노드ID"].map(category_node_best["시설_스냅거리_m"])
    grid_category["도로망_최단거리_m"] = grid_category["노드_시설접근비용_m"] - grid_category["시설_스냅거리_m"]
    grid_category["문화시설_접근거리_m"] = grid_category["격자_스냅거리_m"] + grid_category["노드_시설접근비용_m"]
    grid_category["문화시설_10km_접근가능"] = grid_category["문화시설_접근거리_m"] <= SERVICE_LIMIT_M

    component_facility = facility_component_count[facility_component_count["중분류"] == category][[
        "도로망_연결성분ID", "도로망_연결성분_시설노드수"
    ]]
    grid_category = grid_category.merge(component_facility, on="도로망_연결성분ID", how="left")
    grid_category["도로망_연결성분_시설노드수"] = grid_category["도로망_연결성분_시설노드수"].fillna(0).astype(int)

    target_total = grid_category["문화누리대상자_추정_인구수"].sum()
    target_service = grid_category.loc[grid_category["문화시설_10km_접근가능"], "문화누리대상자_추정_인구수"].sum()

    category_summary_rows.append({
        "중분류": category,
        "분석시설수": len(category_facility),
        "시설노드수": category_node_best.shape[0],
        "접근거리결측_서울격자수": grid_category["문화시설_접근거리_m"].isna().sum(),
        "10km_접근가능_서울격자수": int(grid_category["문화시설_10km_접근가능"].sum()),
        "서울문화누리대상자_총량": int(target_total),
        "10km_서비스권역_문화누리대상자수": int(target_service),
        "서울문화누리대상자_서비스권역비율": target_service / target_total * 100 if target_total > 0 else np.nan,
        "접근거리_평균_m": grid_category["문화시설_접근거리_m"].mean(),
        "접근거리_중앙값_m": grid_category["문화시설_접근거리_m"].median(),
        "접근거리_최대_m": grid_category["문화시설_접근거리_m"].max(),
    })

    category_grid_tables.append(grid_category)
    print(f"{category}: 시설 {len(category_facility):,}개, 시설노드 {category_node_best.shape[0]:,}개, 접근가능 격자 {int(grid_category['문화시설_10km_접근가능'].sum()):,}개, 결측 {grid_category['문화시설_접근거리_m'].isna().sum():,}개")

public_access_grid = gpd.GeoDataFrame(pd.concat(category_grid_tables, ignore_index=True), geometry="geometry", crs="EPSG:5179")
category_access_summary = pd.DataFrame(category_summary_rows)

print("\n서울 격자-중분류 접근성 테이블 구조:", public_access_grid.shape)
print("접근거리 결측:", public_access_grid["문화시설_접근거리_m"].isna().sum())
print("\n중분류별 계산 요약")
display(category_access_summary)


## 행정동·시군구 집계
- 접근성은 행정구역 내 격자의 평균·중앙값·최소·최대 거리로 집계한다.
- 문화누리대상자 수가 있는 격자에는 대상자 가중평균 접근거리를 함께 계산한다.
- 서비스권역 인구비율은 10km 이내 접근 가능한 격자의 문화누리대상자 수를 행정구역 전체 문화누리대상자 수로 나눈다.


In [ ]:

def weighted_mean(value, weight):
    valid = value.notna() & weight.notna()
    if valid.sum() == 0:
        return np.nan
    weight_sum = weight[valid].sum()
    if weight_sum == 0:
        return value[valid].mean()
    return np.average(value[valid], weights=weight[valid])


def aggregate_access(group_cols):
    rows = []
    for keys, group in public_access_grid.groupby(group_cols + ["중분류"]):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: value for col, value in zip(group_cols + ["중분류"], keys)}
        target_total = group["문화누리대상자_추정_인구수"].sum()
        target_service = group.loc[group["문화시설_10km_접근가능"], "문화누리대상자_추정_인구수"].sum()
        row.update({
            "격자수": len(group),
            "문화누리대상자_추정_인구수": int(target_total),
            "10km_서비스권역_문화누리대상자수": int(target_service),
            "문화누리대상자_서비스권역비율": target_service / target_total * 100 if target_total > 0 else np.nan,
            "문화시설_접근거리_평균_m": group["문화시설_접근거리_m"].mean(),
            "문화시설_접근거리_중앙값_m": group["문화시설_접근거리_m"].median(),
            "문화시설_접근거리_최소_m": group["문화시설_접근거리_m"].min(),
            "문화시설_접근거리_최대_m": group["문화시설_접근거리_m"].max(),
            "문화누리대상자_가중평균_접근거리_m": weighted_mean(group["문화시설_접근거리_m"], group["문화누리대상자_추정_인구수"])
        })
        rows.append(row)
    return pd.DataFrame(rows)

public_access_dong = aggregate_access(["시군구", "행정동"])
public_access_gu = aggregate_access(["시군구"])

category_dong_summary = (
    public_access_dong
    .groupby("중분류", as_index=False)
    .agg(
        행정동수=("행정동", "count"),
        평균_서비스권역비율=("문화누리대상자_서비스권역비율", "mean"),
        중앙값_서비스권역비율=("문화누리대상자_서비스권역비율", "median"),
        평균_문화누리대상자_가중평균_접근거리_m=("문화누리대상자_가중평균_접근거리_m", "mean"),
        중앙값_문화누리대상자_가중평균_접근거리_m=("문화누리대상자_가중평균_접근거리_m", "median")
    )
)

print("행정동-중분류 접근성 테이블 구조:", public_access_dong.shape)
print("시군구-중분류 접근성 테이블 구조:", public_access_gu.shape)
print("행정동 서비스권역비율 결측:", public_access_dong["문화누리대상자_서비스권역비율"].isna().sum())
print("시군구 서비스권역비율 결측:", public_access_gu["문화누리대상자_서비스권역비율"].isna().sum())
print("\n중분류별 행정동 요약")
display(category_dong_summary.sort_values("평균_문화누리대상자_가중평균_접근거리_m", ascending=False))


## 접근성·인구비율 산출물 분리
- 접근거리 테이블과 서비스권역 인구비율 테이블을 분리한다.
- 이후 민감도 분석이나 2SFCA·H3SFCA 결과와 비교할 때 지표 의미가 섞이지 않도록 한다.


In [ ]:

grid_access_cols = [
    "GRID_CD", "시도", "시군구", "행정동", "중분류",
    "중심점_x", "중심점_y", "추정_인구수", "문화누리대상자_추정_인구수",
    "문화시설_접근거리_m", "문화시설_10km_접근가능",
    "최근접_가맹점_ID", "최근접_가맹점명", "최근접_가맹점_시도", "최근접_가맹점_시군구",
    "도로망_최단거리_m", "격자_스냅거리_m", "시설_스냅거리_m",
    "도로망_연결성분ID", "도로망_연결성분_node수", "도로망_연결성분_시설노드수", "geometry"
]

grid_access = public_access_grid[grid_access_cols].copy()
grid_ratio = grid_access[["GRID_CD", "시군구", "행정동", "중분류", "문화누리대상자_추정_인구수", "문화시설_10km_접근가능", "geometry"]].copy()

dong_access = public_access_dong[[
    "시군구", "행정동", "중분류", "격자수", "문화누리대상자_추정_인구수",
    "문화시설_접근거리_평균_m", "문화시설_접근거리_중앙값_m", "문화시설_접근거리_최소_m",
    "문화시설_접근거리_최대_m", "문화누리대상자_가중평균_접근거리_m"
]].copy()

dong_ratio = public_access_dong[[
    "시군구", "행정동", "중분류", "격자수", "문화누리대상자_추정_인구수",
    "10km_서비스권역_문화누리대상자수", "문화누리대상자_서비스권역비율"
]].copy()

gu_access = public_access_gu[[
    "시군구", "중분류", "격자수", "문화누리대상자_추정_인구수",
    "문화시설_접근거리_평균_m", "문화시설_접근거리_중앙값_m", "문화시설_접근거리_최소_m",
    "문화시설_접근거리_최대_m", "문화누리대상자_가중평균_접근거리_m"
]].copy()

gu_ratio = public_access_gu[[
    "시군구", "중분류", "격자수", "문화누리대상자_추정_인구수",
    "10km_서비스권역_문화누리대상자수", "문화누리대상자_서비스권역비율"
]].copy()

print("격자 접근거리 테이블:", grid_access.shape)
print("격자 인구비율 기초 테이블:", grid_ratio.shape)
print("행정동 접근거리 테이블:", dong_access.shape)
print("행정동 인구비율 테이블:", dong_ratio.shape)
print("시군구 접근거리 테이블:", gu_access.shape)
print("시군구 인구비율 테이블:", gu_ratio.shape)


## 취약지역 상·하위 20% 추출
- 접근거리는 값이 클수록 취약하다.
- 서비스권역 인구비율은 값이 낮을수록 취약하다.
- 문화누리대상자 추정인구가 0인 행정구역은 취약지역 판정에서 제외한다.


In [ ]:

def make_top_bottom_table(df, unit_cols, metric_col, metric_name, lower_is_vulnerable):
    rows = []
    for category, group in df.groupby("중분류"):
        group = group[group["문화누리대상자_추정_인구수"] > 0].dropna(subset=[metric_col]).copy()
        if len(group) == 0:
            continue
        n_select = max(int(np.ceil(len(group) * 0.2)), 1)
        vulnerable = group.nsmallest(n_select, metric_col).copy() if lower_is_vulnerable else group.nlargest(n_select, metric_col).copy()
        good = group.nlargest(n_select, metric_col).copy() if lower_is_vulnerable else group.nsmallest(n_select, metric_col).copy()
        vulnerable["판정"] = f"{metric_name}_취약상위20pct"
        good["판정"] = f"{metric_name}_양호상위20pct"
        result = pd.concat([vulnerable, good], ignore_index=True)
        result["지표명"] = metric_name
        result["지표값"] = result[metric_col]
        result["중분류내_선정수"] = n_select
        result["순위"] = result.groupby(["중분류", "판정"])["지표값"].rank(method="first", ascending=lower_is_vulnerable).astype(int)
        rows.append(result[unit_cols + ["중분류", "판정", "순위", "지표명", "지표값", "중분류내_선정수", "문화누리대상자_추정_인구수"]])
    return pd.concat(rows, ignore_index=True)


dong_top_bottom = pd.concat([
    make_top_bottom_table(dong_access, ["시군구", "행정동"], "문화누리대상자_가중평균_접근거리_m", "접근거리", lower_is_vulnerable=False),
    make_top_bottom_table(dong_ratio, ["시군구", "행정동"], "문화누리대상자_서비스권역비율", "서비스권역인구비율", lower_is_vulnerable=True)
], ignore_index=True)

gu_top_bottom = pd.concat([
    make_top_bottom_table(gu_access, ["시군구"], "문화누리대상자_가중평균_접근거리_m", "접근거리", lower_is_vulnerable=False),
    make_top_bottom_table(gu_ratio, ["시군구"], "문화누리대상자_서비스권역비율", "서비스권역인구비율", lower_is_vulnerable=True)
], ignore_index=True)

print("행정동 상하위 20% 테이블:", dong_top_bottom.shape)
print("시군구 상하위 20% 테이블:", gu_top_bottom.shape)

print("\n행정동 접근거리 취약상위 20% 예시")
display(dong_top_bottom[dong_top_bottom["판정"] == "접근거리_취약상위20pct"].sort_values(["중분류", "순위"]).head(20))

print("\n행정동 서비스권역인구비율 취약상위 20% 예시")
display(dong_top_bottom[dong_top_bottom["판정"] == "서비스권역인구비율_취약상위20pct"].sort_values(["중분류", "순위"]).head(20))


## 결과 저장
- 격자 산출물은 용량 관리를 위해 GeoParquet과 CSV를 함께 저장한다.
- 행정동·시군구 산출물은 접근거리와 서비스권역 인구비율을 별도 CSV로 저장한다.
- 중분류별 상·하위 20% 판정 결과를 별도 CSV로 저장한다.


In [ ]:

grid_access.to_parquet(NOTEBOOK_OUTPUT_PATH / "공공기관식_최근접접근성_서울격자_중분류별.parquet", index=False)
grid_access.drop(columns="geometry").to_csv(NOTEBOOK_OUTPUT_PATH / "공공기관식_최근접접근성_서울격자_중분류별.csv", index=False, encoding="utf-8-sig")
grid_ratio.drop(columns="geometry").to_csv(NOTEBOOK_OUTPUT_PATH / "공공기관식_서비스권역인구비율_서울격자_중분류별.csv", index=False, encoding="utf-8-sig")
dong_access.to_csv(NOTEBOOK_OUTPUT_PATH / "공공기관식_최근접접근성_서울행정동_중분류별.csv", index=False, encoding="utf-8-sig")
dong_ratio.to_csv(NOTEBOOK_OUTPUT_PATH / "공공기관식_서비스권역인구비율_서울행정동_중분류별.csv", index=False, encoding="utf-8-sig")
gu_access.to_csv(NOTEBOOK_OUTPUT_PATH / "공공기관식_최근접접근성_서울시군구_중분류별.csv", index=False, encoding="utf-8-sig")
gu_ratio.to_csv(NOTEBOOK_OUTPUT_PATH / "공공기관식_서비스권역인구비율_서울시군구_중분류별.csv", index=False, encoding="utf-8-sig")
category_access_summary.to_csv(NOTEBOOK_OUTPUT_PATH / "공공기관식_중분류별_계산요약.csv", index=False, encoding="utf-8-sig")
category_dong_summary.to_csv(NOTEBOOK_OUTPUT_PATH / "공공기관식_중분류별_행정동요약.csv", index=False, encoding="utf-8-sig")
dong_top_bottom.to_csv(NOTEBOOK_OUTPUT_PATH / "공공기관식_서울행정동_상하위20pct_중분류별.csv", index=False, encoding="utf-8-sig")
gu_top_bottom.to_csv(NOTEBOOK_OUTPUT_PATH / "공공기관식_서울시군구_상하위20pct_중분류별.csv", index=False, encoding="utf-8-sig")

print("저장 완료")
for file_path in sorted(NOTEBOOK_OUTPUT_PATH.glob("공공기관식_*")):
    print(file_path.name)


## 최종 검토
- 서울 격자 기준 중분류별 접근거리와 서비스권역 인구비율이 산출되었는지 확인한다.
- 행정동·시군구별 집계에서 문화누리대상자 총량이 보존되는지 확인한다.
- 접근거리 결측은 도로망 연결 성분 또는 해당 중분류 시설 부재에서 발생하는지 확인한다.


In [ ]:

print("격자 산출물 중분류 수:", grid_access["중분류"].nunique())
print("격자 산출물 GRID_CD 수:", grid_access["GRID_CD"].nunique())
print("행정동 산출물 행정동 수:", dong_access[["시군구", "행정동"]].drop_duplicates().shape[0])
print("시군구 산출물 시군구 수:", gu_access["시군구"].nunique())

mnc_grid_total = grid_seoul["문화누리대상자_추정_인구수"].sum()
mnc_dong_total_check = dong_ratio[dong_ratio["중분류"] == facility_mid_categories[0]]["문화누리대상자_추정_인구수"].sum()
mnc_gu_total_check = gu_ratio[gu_ratio["중분류"] == facility_mid_categories[0]]["문화누리대상자_추정_인구수"].sum()

print("서울 격자 문화누리대상자 총량:", mnc_grid_total)
print("행정동 집계 총량 점검:", mnc_dong_total_check)
print("시군구 집계 총량 점검:", mnc_gu_total_check)
print("행정동 총량 오차:", mnc_grid_total - mnc_dong_total_check)
print("시군구 총량 오차:", mnc_grid_total - mnc_gu_total_check)

print("\n접근거리 결측 중분류별")
print(grid_access.groupby("중분류")["문화시설_접근거리_m"].apply(lambda x: x.isna().sum()).reindex(facility_mid_categories))

print("\n서비스권역 인구비율 요약")
display(category_dong_summary.sort_values("평균_서비스권역비율"))


## 실행 결과 요약
- 전체 분석권역 격자: 503,586개, 서울 결과대상 격자: 60,528개
- 전체 분석권역 가맹점: 11,352개
- 도로망 그래프: node 97,231개, edge 121,120개, 연결 성분 31개
- 서울 격자 스냅거리 500m 초과: 8,334개, 가맹점 스냅거리 300m 초과: 244개
- 접근거리 결측: 스포츠관람 7,757개, 그 외 중분류 각 14개
- 10km 서비스권역 인구비율: 스포츠관람 83.38%, 그 외 중분류 99.97%
- 평균 접근거리: 스포츠관람 5,543m, 관광지 2,791m, 음악 2,662m, 공연 2,322m, 영상 2,236m, 체육용품 1,557m, 도서 1,320m, 문화체험 1,296m, 미술 1,214m, 체육시설 1,092m
- 해석: 정부식 10km 기준은 대부분 중분류에서 매우 완화적이며, 서비스권역 인구비율은 스포츠관람을 제외하면 변별력이 약하다. 중분류별 취약지역 비교에는 최근접 접근거리 지표가 더 유효하다.


## 보고서 기입용 정리

공공기관식 문화시설 접근성 지표는 격자 단위 거주지에서 가장 가까운 문화시설까지의 도로망 최단거리로 정의하였다. 정부·공공기관 지표에서 공연·문화시설은 지역거점시설로 분류되며, 차량 약 20분 또는 10km 이내 도달 가능성을 서비스권역 기준으로 사용한다. 이에 따라 본 분석에서는 서울 100m 격자를 기준으로 중분류별 최근접 문화누리 가맹점까지의 도로망 최단거리를 계산하고, 10km 이내 접근 가능한 격자의 문화누리대상자 추정인구 비율을 행정동 및 시군구 단위로 집계하였다.

\[
A_{i,k}=\min_{j \in C_k}(d_{i,n_i}+d_{n_i,n_j}+d_{n_j,j})
\]

\[
R_{r,k}=\frac{\sum_{i \in r}P_i \cdot \mathbf{1}(A_{i,k}\leq 10{,}000)}{\sum_{i \in r}P_i}\times100
\]

여기서 $A_{i,k}$는 격자 $i$에서 중분류 $k$ 시설까지의 최근접 접근거리, $P_i$는 격자별 문화누리대상자 추정인구, $R_{r,k}$는 행정구역 $r$의 중분류별 서비스권역 내 대상자 비율이다. 원 정부지표가 전체 거주인구를 기준으로 하는 데 비해, 본 프로젝트에서는 문화누리카드 정책 대상자의 실질적 접근성을 평가하기 위해 문화누리대상자 추정인구를 분모와 분자에 적용하였다.

지역문화지수는 문화정책, 문화자원, 문화활동, 문화향유의 4대 영역별 지표값을 Z-score로 표준화한 뒤, 전문가 AHP 분석으로 도출한 가중치를 적용해 합산하는 방식이다. 2023년 기준 지역문화지수 분석에서는 대분류 가중치를 문화정책 0.314, 문화자원 0.226, 문화활동 0.180, 문화향유 0.280으로 설정하였다. AHP 가중치는 전문가의 쌍대비교를 통해 영역 간 상대적 중요도를 산정하고, 일관성 검정을 통과한 응답을 기준으로 최종 산출한 값이다.


## 우리 도보·대중교통 기준 최근접 접근성 추가
- 목적: 정부식 최근접 시설 알고리즘을 유지하되, 서비스권역 반경을 10km 차량 기준이 아니라 우리 분석의 도보·대중교통 기준으로 바꿔 비교함.
- 도보 기준: 도서, 문화체험, 음악, 영상, 체육시설, 체육용품은 도보 750m 이내 접근 가능 여부로 판정함.
- 대중교통 기준: 미술, 공연, 스포츠관람, 관광지는 대중교통 20분 이내 접근 가능 여부로 판정함.
- 해석: 최근접 거리 알고리즘 자체가 우리 접근성 분석 기준에서도 유사한 취약지역을 식별하는지 검토함.

In [ ]:
# 이 셀은 공공기관식 최근접 접근성의 우리 반경 버전 산출물을 확인한다.
OUR_RADIUS_PATH = NOTEBOOK_OUTPUT_PATH / "공공기관식_우리반경_최근접접근성_서울시군구_중분류별.csv"
COMPARE_PATH = NOTEBOOK_OUTPUT_PATH / "공공기관식_정부10km_vs_우리반경_서울시군구_중분류별.csv"

public_our_gu = pd.read_csv(OUR_RADIUS_PATH)
public_compare_gu = pd.read_csv(COMPARE_PATH)

print("우리 반경 시군구 결과:", public_our_gu.shape)
print("정부10km vs 우리반경 비교:", public_compare_gu.shape)
display(public_our_gu.head())
display(public_compare_gu.sort_values("우리반경_minus_정부10km_비율차").head(10))

### 우리 반경 기준 주요 결과
- 격자 산출물: `공공기관식_우리반경_최근접접근성_서울격자_중분류별.csv`
- 행정동 산출물: `공공기관식_우리반경_최근접접근성_서울행정동_중분류별.csv`
- 시군구 산출물: `공공기관식_우리반경_최근접접근성_서울시군구_중분류별.csv`
- 비교 산출물: `공공기관식_정부10km_vs_우리반경_서울시군구_중분류별.csv`
- 정부 10km 기준은 대부분 100%에 가까워 변별력이 낮았고, 우리 반경 기준은 도보·대중교통 생활권의 접근 가능 여부를 더 엄격하게 보여줌.